# Проект: Рынок видеоигр

Автор: Минько Никита

Дата: 2026-08-01

В рамках проекта мы поможем коллегам из «Секретов Темнолесья» изучить развитие игровой индустрии. Мы будем работать с историческими данными, которые коллеги собрали из открытых источников. В них есть информация о продажах игр, сделанных в разных жанрах и выпущенных на разных платформах, а также пользовательские и экспертные оценки игр.

Наша задача - познакомиться с данными, проверить их корректность и провести предобработку, получив необходимый срез данных.

Нам необходимо сделать следующее:
- Отобрать данные по времени выхода игры. Нам нужен период с 2000 по 2013 год включительно.
- Категоризовать игры по оценкам пользователей и экспертов. Выделить три категории:
  - `высокая оценка` — с оценкой от 8 до 10 и от 80 до 100, включая правые границы интервалов;
  - `средняя оценка` — с оценкой от 3 до 8 и от 30 до 80, не включая правые границы интервалов;
  - `низкая оценка` — с оценкой от 0 до 3 и от 0 до 30, не включая правые границы интервалов.
- Выделить топ-7 платформ по количеству игр, выпущенных за весь требуемый период.

## Загрузка и знакомство с данными

Импортируем библиотеку pandas:

In [1]:
import pandas as pd

Загружаем наш датасет и превращаем его в датафрейм:

In [2]:
# Загружаем датасет по ссылке из описания проекта:
df = pd.read_csv('https://********/********/new_games.csv')

Чтобы убедиться, что у нас получилось выгрузить датасет и сформировать датафрейм, выведем первые 5 строк. Также посмотрим общую сводку о структуре датафрейма, используя метод `info()`.

In [3]:
# Выводим первые 5 строк датафрейма:
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [4]:
# Используем метод info() для вывода общей сводки о датафрейме:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


По полученным данным можем сформировать следующий вывод:

Датасет `new_games.csv` содержит 11 столбцов и 16956 строк, в которых представлены исторические данные о продажах игр, сделанных в разных жанрах и выпущенных на разных платформах, а также пользовательские и экспертные оценки игр. Столбцы отображают содержимое. Данные соответствуют описанию.

Изучим типы данных и их корректность:
- **Числовые значения с плавающей запятой (float64).** Четыре столбца имеют тип данных `float64`:
    - `Year of Release` содержит год выпуска. Это неверное решение, так как значение не должно включать дробные значения. Для отображения года в нашем исследовании стоило бы привести значение к целочисленному типу данных `int64`. Также можно было бы выбрать `Int16`. Выбор такой разрядности был бы обусловлен тем, что она сэкономит память и при этом вместит все возможные значения года. Однако в рамках исследования это не критичное допущение.
    - `NA sales` содержит продажи в Северной Америке (в миллионах проданных копий). Выбор типа данных оптимальный, так как дробная часть будет вмещать себе количество продаж до миллиона. Т.е., например, один миллион и триста тысяч продаж. Эти триста тысяч будут отображены в дробной части.
    - `Other sales` содержит продажи в других странах (в миллионах проданных копий). По аналогии с `NA sales` выбор оптимальный.
    - `Critic Score` содержит оценку критиков (от 0 до 100). По первым пяти строкам данных мы видим, что столбец содержит целые числа, однако для сохранения дробной части, если она присутствует в последующих строках, оптимально оставить тип данных `float64`.
- **Строковые данные (str или object).** Семь столбцов имеют тип данных `str`:
    - `Name` содержит название игры, что соответсвует строковым (текстовым) данным.
    - `Platform`, `Genre` и `Rating` содержат информацию о платформе, жанре и рейтинге организации ESRB. Так как мы имеем заранее определенный набор данных (перечни платформ, жанров и отметок рейтинга ограничены и заранее известны), можно было бы использовать тип `category`, чтобы оптимизировать память. Однако в рамках исследования оставим всё же тип данных `str`, так как, вероятно, столбцы могут содержать неявные дубликаты. Да и в рамках исследования это не будет иметь значения.
    - `EU sales`, `JP sales` содержат продажи в Европе и Японии соответственно (в миллионах проданных копий). По аналогии с `NA sales` и `Other sales` оптимально перевести в тип `float64`.
    - `User Score` содержит оценку пользователей (от 0 до 10). По аналогии с `Critic Score` стоит перевести в тип `float64`.

Заметим, что в столбцах встречаются пропуски. Посчитаем их абсолютное и относительное количество:

In [5]:
# Считаем абсолютное значение (их сумму по столбцам):
missing_sum = df.isna().sum()
# Считаем относительное значение (долю) и переводим в проценты:
missing_share = df.isna().mean() * 100
# Округлим относительное значение:
missing_share = missing_share.round(2)
# Создадим датафрейм с пропусками для наглядности:
missing_df = pd.DataFrame({'Количество пропусков': missing_sum, 'Доля пропусков в процентах': missing_share})
# Выведем датафрейм:
missing_df

,Количество пропусков,Доля пропусков в процентах
Name,2,0.01
Platform,0,0.00
Year of Release,275,1.62
Genre,2,0.01
NA sales,0,0.00
EU sales,0,0.00
JP sales,0,0.00
Other sales,0,0.00
Critic Score,8714,51.39
User Score,6804,40.13


По пропускам мы наблюдаем следующую ситуацию. Пропуски присутствуют в столбцах:
- `Name` - незначительное кол-во пропусков, равное 2 (0,01% от данных в столбце);
- `Year of Release` - 275 пропусков (1,62% от данных в столбце)
- `Genre` - незначительное кол-во пропусков, равное 2 (0,01% от данных в столбце);
- `Critic Score` - значительное кол-во пропусков, равное 8714 (51,39% от данных в столбце);
- `User Score` - значительное кол-во пропусков, равное 6804 (40,13% от данных в столбце);
- `Rating` - значительное кол-во пропусков, равное 6871 (40,52% от данных в столбце).

Но это результат до предобработки. К пропускам мы вернемся чуть позже.

## Проверка ошибок в данных и их предобработка

#### Названия, или метки, столбцов датафрейма

Для начала выведем названия столбцов нашего датафрейма:

In [6]:
df.columns.tolist()

['Name',
 'Platform',
 'Year of Release',
 'Genre',
 'NA sales',
 'EU sales',
 'JP sales',
 'Other sales',
 'Critic Score',
 'User Score',
 'Rating']

Приведем названия столбцов к "питоновскому" варианту `snake case`, т.е. сделаем наш текст в нижнем регистре и заменим пробелы нижным подчеркиванием:

In [7]:
# Создадим список, в который позже мы добавим приведенные названия столбцов:
snake_columns = []
# Создадим цикл, с помощью которого проведем нужные операции с форматированием текста:
for column in df.columns:
    # Делаем регистр нижним:
    column = column.lower()
    # Заменяем пробелы на нижнее подчеркивание:
    column = column.replace(' ', '_')
    # Приведенное название столбца добавляем в наш список:
    snake_columns.append(column)

# Заменяем в нашем датафрейм названия столбцов:
df.columns = snake_columns
# Выводим наши "snake_case"-столбцы:
df.columns.tolist()

['name',
 'platform',
 'year_of_release',
 'genre',
 'na_sales',
 'eu_sales',
 'jp_sales',
 'other_sales',
 'critic_score',
 'user_score',
 'rating']

Готово. Названия столбцов приведены к стилю `snake case`.

#### Типы данных

Про используемые типы данных в столбцах мы уже писали выше. Там же мы определили, в каких столбцах нужно сменить тип: `year_of_release`, `eu_sales`, `jp_sales`, `user_score`. Сейчас приведем эти столбцы к релевантному типу.

Вероятно, столбцы могут содержать строковые значения. Выбранные столбцы должны содержать числовую информацию, поэтому приведем их значения к числам. Используем `pd.to_numeric` для этого. К нему применим параметр `errors='coerce'`, который нечисловые значения заменит на пропуск `NaN`.

In [8]:
# Приводим значения столбцов к числам:
df[['year_of_release', 'eu_sales', 'jp_sales', 'user_score']] = df[['year_of_release', 'eu_sales', 'jp_sales', 'user_score']].apply(pd.to_numeric, errors='coerce')

Столбец `year_of_release` мы собираемся приводить к целочисленному типу. Помним, что столбцы с числовыми данными и пропусками нельзя преобразовать к типу `int64`. В нашем датасете данный столбец имеет пропуски, как мы отмечали ранее. Их доля составляется 1,62%. Процент невелик, поэтому можем удалить эти пропуски. Это значительно не отразится на результатах исследования.

In [9]:
# Удаляем пропуски из столбца 'year_of_release':
df = df.dropna(subset = ['year_of_release'])

Сейчас можем продолжить приведение наших столбцов к нужному типу данных:

In [10]:
# Приводим столбцы к релевантному типу данных:
df['year_of_release'] = df['year_of_release'].astype('int64')
df['eu_sales'] = df['eu_sales'].astype('float64')
df['jp_sales'] = df['jp_sales'].astype('float64')
df['user_score'] = df['user_score'].astype('float64')
# Выведем сводную информацию, чтобы убедиться, что тип был изменен:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16681 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16679 non-null  object 
 1   platform         16681 non-null  object 
 2   year_of_release  16681 non-null  int64  
 3   genre            16679 non-null  object 
 4   na_sales         16681 non-null  float64
 5   eu_sales         16675 non-null  float64
 6   jp_sales         16677 non-null  float64
 7   other_sales      16681 non-null  float64
 8   critic_score     8085 non-null   float64
 9   user_score       7558 non-null   float64
 10  rating           9901 non-null   object 
dtypes: float64(6), int64(1), object(4)
memory usage: 1.5+ MB


Тип данных был успешно изменен. Строковые значения были заменены на пропуски. Также отметим, что в результате работы с типами данных было удалено 275 строк.

#### Наличие пропусков в данных

Снова посчитаем количество пропусков в каждом столбце в абсолютных и относительных значениях. После приведения к нужным типам данных и замены строковых значений на пропуски результаты будут отличаться от тех, что были на этапе первоначального знакомства с датасетом.

Итак, приступим к подсчету по уже известному нам способу:

In [11]:
# Считаем абсолютное значение (их сумму по столбцам):
new_missing_sum = df.isna().sum()
# Считаем относительное значение (долю) и переводим в проценты:
new_missing_share = df.isna().mean() * 100
# Округлим относительное значение:
new_missing_share = new_missing_share.round(2)
# Создадим датафрейм с пропусками для наглядности:
new_missing_df = pd.DataFrame({'Количество пропусков': new_missing_sum, 'Доля пропусков в процентах': new_missing_share})
# Выведем датафрейм:
new_missing_df

,Количество пропусков,Доля пропусков в процентах
name,2,0.01
platform,0,0.00
year_of_release,0,0.00
genre,2,0.01
na_sales,0,0.00
eu_sales,6,0.04
jp_sales,4,0.02
other_sales,0,0.00
critic_score,8596,51.53
user_score,9123,54.69


По пропускам мы наблюдаем следующую ситуацию. Пропуски присутствуют в столбцах:
- `name` — незначительное кол-во пропусков, равное 2 (0,01% от данных в столбце);
- `genre` — незначительное кол-во пропусков, равное 2 (0,01% от данных в столбце);
- `eu_sales` — незначительное кол-во пропусков, равное 6 (0,04% от данных в столбце);
- `jp_sales` — незначительное кол-во пропусков, равное 4 (0,02% от данных в столбце);
- `critic_score` — значительное кол-во пропусков, равное 8596 (51,53% от данных в столбце);
- `user_score` — значительное кол-во пропусков, равное 9123 (54,69% от данных в столбце);
- `rating` — значительное кол-во пропусков, равное 6780 (40,65% от данных в столбце).

Пропуски отмечены в 7 столбцах.

Незначительное значение отмечено в столбцах `name`, `genre`, `eu_sales`, `jp_sales` - значения менее 1%. Возможно, данные были утеряны при загрузке в базу. Их стоит удалить. Потеря такого количества строк на результаты дальнейшего анализа не повлияет.

In [12]:
# Создадим список с названиями столбцов, в которых будем удалять пропуски:
columns_for_clean = ['name', 'genre', 'eu_sales', 'jp_sales']
# Создадим цикл для удаления пропусков в столбах:
for column_for_clean in columns_for_clean:
    df = df.dropna(subset=[column_for_clean])

Максимальное и даже критическое значение пропусков, как мы видим, зафиксировано в `critic_score`, `user_score`, `rating`. Вероятно, это связано с тем, что некоторые редкие и непопулярные игры не были размещены на платформах для оценки игроками и критиками, а также не были интересны ESRB. Такие значения не стоит удалять, так как мы потеряем половину нашего датасета и исказим результаты последующего анализа. Эти пропуски можно было бы заполнить значениями-индикаторами (для оценок можно было бы выбрать какое-то отрицательное значение, а для рейтинга что-то в стиле `Нет данных / no data` или `Неизвестен / Unknown`. Однако в рамках нашего исследования просто оставим эти пропуски без изменений.

In [13]:
# Выведем сводную информацию о нашем датафрейме:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16669 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16669 non-null  object 
 1   platform         16669 non-null  object 
 2   year_of_release  16669 non-null  int64  
 3   genre            16669 non-null  object 
 4   na_sales         16669 non-null  float64
 5   eu_sales         16669 non-null  float64
 6   jp_sales         16669 non-null  float64
 7   other_sales      16669 non-null  float64
 8   critic_score     8077 non-null   float64
 9   user_score       7550 non-null   float64
 10  rating           9892 non-null   object 
dtypes: float64(6), int64(1), object(4)
memory usage: 1.5+ MB


Отметим, что после работы с пропусками было удалено 12 строк.

#### Явные и неявные дубликаты в данных

Изучим уникальные значения в категориальных данных. Они представлены в столбцах `platform`, `genre`, `rating`.

Начнем с платформ.

In [14]:
# Выведем уникальные значения платформ:
unique_platform = df['platform'].unique()
unique_platform

array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

Среди платформ представлено 31 значение и все они уникальны. Явных или неявных дубликатов тут нет.

Теперь проверим уникальность жанров.

In [15]:
# Выведем уникальные значения жанров:
unique_genre = df['genre'].unique()
unique_genre

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION', 'SHOOTER',
       'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE', 'SIMULATION',
       'PUZZLE', 'STRATEGY'], dtype=object)

Что касается жанров, наблюдаем здесь неявные дубликаты. Связаны они с использованием разных регистров букв. Сделаем для всех жанров одинаковый нижний регистр и выведем снова список уникальных жанров. 

In [16]:
# Изменим регистр на нижний:
df['genre'] = df['genre'].str.lower()

# Снова выведем уникальные значения жанров:
unique_genre = df['genre'].unique()
unique_genre

array(['sports', 'platform', 'racing', 'role-playing', 'puzzle', 'misc',
       'shooter', 'simulation', 'action', 'fighting', 'adventure',
       'strategy'], dtype=object)

Готово. Так мы устранили неявные дубликаты в столбце с жанрами и получили 12 уникальных значений.

Теперь перейдем к рейтингу.

In [17]:
# Выведем уникальные значения рейтинга:
unique_rating = df['rating'].unique()
unique_rating

array(['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP'], dtype=object)

Среди рейтинга мы наблюдаем 9 уникальных значений (1 из них - пропуск, который мы решили оставить).

Теперь проверим названия игр, но выведем не уникальные значения, а их количество. Такое отличие заключается в том, что уникальных значений предположительно слишком много.

In [18]:
# Выведем количество уникальных значений названий игр:
unique_name_count = df['name'].nunique()
unique_name_count

11422

In [19]:
# Теперь приведем все названия к верхнему регистру и повторим:
df['name'] = df['name'].str.upper()
unique_name_count = df['name'].nunique()
unique_name_count

11422

Неявных дубликатов не выявлено. Однако заметим, что это число не равняется длине датафрейма. Это обусловлено тем, что некоторые игры были выпущены для нескольких платформ. Таким образом названия некоторых игр встречаются несколько раз в столбце.

Давайте теперь разберемся с явными дубликатами. Посчитаем их количество и узнаем, есть ли они вообще.

In [20]:
# Считаем количество явных дубликатов во всем датафрейме:
print(df.duplicated().sum())

235


Итого мы наблюдаем 235 строк дубликатов. Удалим их.

In [21]:
# Удаляем дубликаты:
df = df.drop_duplicates()
# Выведем сводную информацию о нашем датафрейме:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16434 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16434 non-null  object 
 1   platform         16434 non-null  object 
 2   year_of_release  16434 non-null  int64  
 3   genre            16434 non-null  object 
 4   na_sales         16434 non-null  float64
 5   eu_sales         16434 non-null  float64
 6   jp_sales         16434 non-null  float64
 7   other_sales      16434 non-null  float64
 8   critic_score     7975 non-null   float64
 9   user_score       7455 non-null   float64
 10  rating           9759 non-null   object 
dtypes: float64(6), int64(1), object(4)
memory usage: 1.5+ MB


Мы избавились от дубликатов.

#### Выводы предобработки

Таким образом, мы провели проверку ошибок и предобработку данных.
1. **Названия, или метки, столбцов датафрейма**. В ходе проведения предобработки данных названия столбцов были приведены к стилю `snake case`.
2. **Типы данных**. В результате предобработки данных мы привели столбцы `year_of_release`, `eu_sales`, `jp_sales`, `user_score` к соответствующим числовым типам данных. В столбцах присутствовали строковые значения, которые мы заменили на пропуски. При переводе `year_of_release` к целочисленному типу, пропуски данного столбца были удалены.
3. **Наличие пропусков в данных**. Пропуски были отмечены в 7 столбцах. Незначительное значение отмечено в столбцах `name`, `genre`, `eu_sales`, `jp_sales` (0,01%, 0,01%, 0,04% и 0,02% соответственно) - значения менее 1%. Их удаление не влияет на результаты будущего анализа, поэтому было принято соответствующее решение.
Максимальное и даже критическое значение пропусков было зафиксировано в `critic_score`, `user_score`, `rating` (51,53%, 54,69% и 40,65% соответственно). Пропуски занимали порядка половины датафрейма. Их удаление исказило бы результаты будущего анализа, поэтому было принято решение сохранить их.
4. **Явные и неявные дубликаты в данных**. В ходе предобработки данных мы решили изучить уникальные значения в категориальных данных, представленных в столбцах `platform`, `genre`, `rating`. Как результат неявные дубликаты были отмечены в столбце `genre`. Была проведена нормализацию данных с текстовыми значениями путем смены регистра на нижний.
Также была проведена проверка на явные дубликаты. Их было обнаружено 235. Мы от них избавились.

В завершение этого пункта давайте также "посчитаем потери". Вспомним, что на этапе знакомства с данными у нас было 16956 строк. В ходе предобработки данных:
- Было удалено 275 строк с пропусками, когда приводили столбец с годом выпуска к целочисленному формату;
- Было удалено 12 строк при работе с пропусками;
- Было удалено 235 строк при работе с дубликатами.

Итого в рамках преобработки было удалено 522 строки, что равняется 3,08% от исходного количества строк в датасете.

## Фильтрация данных

В дальнейшем исследовании мы хотим изучить историю продаж игр в начале XXI века, и нас интересует период с 2000 по 2013 год включительно.  Отобранные данные по этому срезы сохраним в отдельный датафрейм `df_actual`.

In [22]:
# Фильтруем данные по столбцу 'year_of_release', устанавливаем верхнюю и нижнюю границу
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)]
# Давайте проверим, включает ли наш срез 2013 год, отсортировав данные по убыванию:
df_actual = df_actual.sort_values(by='year_of_release', ascending = False)
# Выведем первые 5 строк:
df_actual.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
15817,MEIJI TOUKYOU RENKA,PSP,2013,adventure,0.00,0.0,0.02,0.00,NaN,NaN,NaN
7233,ZUMBA FITNESS: WORLD PARTY,Wii,2013,misc,0.11,0.1,0.00,0.02,NaN,NaN,E
9501,HYPERDIMENSION NEPTUNIA,PSV,2013,role-playing,0.05,0.0,0.07,0.01,NaN,NaN,NaN
15336,STEINS;GATE: SENKEI KOUSOKU NO PHONOGRAM,X360,2013,adventure,0.00,0.0,0.02,0.00,NaN,NaN,NaN
11558,PRETTY RHYTHM: MY DECO RAINBOW WEDDING,3DS,2013,misc,0.00,0.0,0.08,0.00,NaN,NaN,NaN


In [23]:
# Теперь проверим, включает ли наш срез 2000 год, отсортировав данные по возрастанию:
df_actual = df_actual.sort_values(by='year_of_release')
# Выведем первые 5 строк:
df_actual.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
8212,THE ADVENTURES OF COOKIE & CREAM,PS2,2000,puzzle,0.09,0.07,0.0,0.02,NaN,NaN,NaN
12900,ANIMORPHS: SHATTERED REALITY,PS,2000,action,0.03,0.02,0.0,0.00,27.0,NaN,E
7424,WILD WILD RACING,PS2,2000,racing,0.10,0.08,0.0,0.03,64.0,NaN,E
7472,HBO BOXING,PS,2000,fighting,0.12,0.08,0.0,0.01,26.0,4.0,E
3620,NASCAR RUMBLE,PS,2000,racing,0.31,0.21,0.0,0.04,NaN,NaN,NaN


Готово. Мы отфильтровали данные и создали новый срез. Наш новый датафрейм включает данные за 2000-2013 годы.

## Категоризация данных

Теперь для оптимизации последующего анализа проведем категоризацию данных. Далее выведем топ-7 платформ по количеству выпущенных игр.
1. Разделим все игры по оценкам пользователей и выделим такие категории:
   - высокая оценка (от 8 до 10 включительно);
   - средняя оценка (от 3 до 8, не включая правую границу интервала);
   - низкая оценка (от 0 до 3, не включая правую границу интервала).

In [24]:
# Создадим границы для категорий.
# 10.1 - чтобы включить 10, так как включаться будет только левая граница (будем использовать right = False в pd.cut):
user_score_bins = [0, 3, 8, 10.1]
# Поименуем сами категории:
user_score_labels = ['низкая оценка', 'средняя оценка', 'высокая оценка']
# Создаем столбец с категориями. Категорируем по нашим границам:
df_actual['user_score_category'] = pd.cut(df_actual['user_score'], bins = user_score_bins, labels = user_score_labels, right = False)

In [25]:
user_score_stat = df_actual.groupby('user_score_category')['name'].count()
user_score_stat

user_score_category
низкая оценка      116
средняя оценка    4078
высокая оценка    2282
Name: name, dtype: int64

2. Разделим все игры по оценкам критиков и выделим такие категории:
   - высокая оценка (от 80 до 100 включительно);
   - средняя оценка (от 30 до 80, не включая правую границу интервала);
   - низкая оценка (от 0 до 30, не включая правую границу интервала).

In [26]:
# Создадим границы для категорий.
# 100.1 - по аналогии с 'user_score_bins':
critic_score_bins = [0, 30, 80, 100.1]
# Поименуем сами категории:
critic_score_labels = ['низкая оценка', 'средняя оценка', 'высокая оценка']
# Создаем столбец с категориями. Категорируем по нашим границам:
df_actual['critic_score_category'] = pd.cut(df_actual['critic_score'], bins = critic_score_bins, labels = critic_score_labels, right = False)

Посчитаем количество игр по категориям оценок критиков и выведем результат.

In [27]:
critic_score_stat = df_actual.groupby('critic_score_category')['name'].count()
critic_score_stat

critic_score_category
низкая оценка       55
средняя оценка    5421
высокая оценка    1686
Name: name, dtype: int64

Проверим, что у нас получилось. Выведем 15 первых строк столбцов `name`, `platform`, `user_score`, `user_score_category`, `critic_score` и `critic_score_category`.

In [28]:
df_actual[['name', 'platform', 'user_score', 'user_score_category', 'critic_score', 'critic_score_category']].head(15)

,name,platform,user_score,user_score_category,critic_score,critic_score_category
8212,THE ADVENTURES OF COOKIE & CREAM,PS2,NaN,NaN,NaN,NaN
12900,ANIMORPHS: SHATTERED REALITY,PS,NaN,NaN,27.0,низкая оценка
7424,WILD WILD RACING,PS2,NaN,NaN,64.0,средняя оценка
7472,HBO BOXING,PS,4.0,средняя оценка,26.0,низкая оценка
3620,NASCAR RUMBLE,PS,NaN,NaN,NaN,NaN
7499,CRUIS'N EXOTICA,N64,NaN,NaN,NaN,NaN
12907,G1 JOCKEY 2,PS2,NaN,NaN,NaN,NaN
7523,POPOLOCROIS MONOGATARI II,PS,NaN,NaN,NaN,NaN
7550,ESPN WINTER X GAMES: SNOWBOARDING 2002,PS2,7.9,средняя оценка,64.0,средняя оценка
7624,INTERNATIONAL SUPERSTAR SOCCER 2000,PS2,NaN,NaN,NaN,NaN


Итак, у нас получилось провести категоризацию данных по оценкам игроков и по оценкам критиков, присвоив категории с низкой, средней и высокой оценками.
- По мнению пользователей `высокую оценку` получили 2282 игры, `среднюю` - 4078, `низкую` - 116.
- По мнению критиков `высокую оценку` получили 1686 игр, `среднюю` - 5421, `низкую` - 55.

3. Теперь давайте создадим топ-7 платформ по количеству игр.

In [29]:
# Прогруппируем данные по названию платформы и посчитаем кол-во строк:
top_platform = df_actual.groupby('platform')['name'].count()
# Отсортируем по убыванию:
top_platform = top_platform.sort_values(ascending = False)
# Выведем первые 7 строк:
top_platform.head(7)

platform
PS2     2126
DS      2117
Wii     1275
PSP     1179
X360    1118
PS3     1087
GBA      810
Name: name, dtype: int64

Готово. Рейтинг платформ по количеству выпущенных игр составлен.

## Общий вывод

Выполнение проекта было разделено на 4 этапа.
1. **Загрузка и знакомство с данными**. На этом этапе мы выгрузили наш изначальный датасет, осмотрели его, оценили масштабы (11 столбцов и 16956 строк) и сформировали первые выводы до проведения предобработки. Было определено в каких столбцах верно установлен тип данных (`Name`, `Platform`, `Genre`, `Rating` - строковый тип; `NA sales`, `Other sales`, `Critic Score` - числовой с плавающей запятой), а в каких требуется корректировка (`Year of Release` - было числовой с плавающей запятой, а требовался целочисленный тип; `EU sales`, `JP sales`, `User Score` - был строковый, а требовался числовой тип с плавающей запятой). Также мы посчитали количество пропусков до предобработки:
   - `Name` - незначительное кол-во пропусков, равное 2 (0,01% от данных в столбце);
   - `Year of Release` - 275 пропусков (1,62% от данных в столбце)
   - `Genre` - незначительное кол-во пропусков, равное 2 (0,01% от данных в столбце);
   - `Critic Score` - значительное кол-во пропусков, равное 8714 (51,39% от данных в столбце);
   - `User Score` - значительное кол-во пропусков, равное 6804 (40,13% от данных в столбце);
   - `Rating` - значительное кол-во пропусков, равное 6871 (40,52% от данных в столбце).
2. **Проверка ошибок в данных и их предобработка**. Как это было:
   - В ходе проведения предобработки данных названия столбцов были приведены к стилю `snake case`.
   - Также мы привели столбцы `year_of_release`, `eu_sales`, `jp_sales`, `user_score` к соответствующим числовым типам данных. Текстовые значения мы заменили на пропуски. При переводе `year_of_release` к целочисленному типу пропуски данного столбца были удалены.
   - Что касается остальных пропусков, они были отмечены в 7 столбцах. Незначительное значение было отмечено в столбцах `name`, `genre`, `eu_sales`, `jp_sales` (0,01%, 0,01%, 0,04% и 0,02% соответственно) - значения менее 1%. Их удаление не влияет на результаты будущего анализа, поэтому было мы от них избавились путем удаления. Максимальное и даже критическое значение пропусков было зафиксировано в `critic_score`, `user_score`, `rating` (51,53%, 54,69% и 40,65% соответственно). Пропуски занимали порядка половины датафрейма. Такое количество пропусков может быть обосновано рядом причин, среди которых, например, можно отметить непопулярность и редкость некоторых игр. Удаление таких пропусков исказит результаты будущего анализа, поэтому было принято решение сохранить их.
   - В ходе предобработки данных мы решили изучить уникальность значения в категориальных данных, представленных в столбцах `platform`, `genre`, `rating`. Как результат неявные дубликаты были отмечены в столбце `genre`. Была проведена нормализацию данных с текстовыми значениями путем смены регистра на нижний. Также была проведена проверка на явные дубликаты. Их было обнаружено 235. Мы от них избавились.

    Как итог в рамках предобработки мы удалили 522 строки, что равняется 3,08% от исходного количества строк в датасете.
3. **Фильтрация данных**. Перед нами стояла задача отобрать данные по времени выхода игры. Нас интересовал период с 2000 по 2013 гг. включительно. Данные были отфильтрованы и наш срез был сохранен в новый датафрейм с названием `df_actual`.
4. **Категоризация данных**. Мы провели категоризацию игр по оценкам пользователей и экспертов. Выделили три категории:
   - `высокая оценка` — с оценкой от 8 до 10 и от 80 до 100, включая правые границы интервалов;
   - `средняя оценка` — с оценкой от 3 до 8 и от 30 до 80, не включая правые границы интервалов;
   - `низкая оценка` — с оценкой от 0 до 3 и от 0 до 30, не включая правые границы интервалов.

    Так, по оценкам пользователей сформировалась такая статистика: `высокую оценку` получили 2282 игры, `среднюю` - 4078, `низкую` - 116. По мнению критиков `высокую оценку` получили 1686 игр, `среднюю` - 5421, `низкую` - 55.

   Так же был составлен топ-7 платформ по количеству выпущенных игр в рамках изучаемого среза. Результат такой:
   - *1-е место*. `PS2` - 2126 игр;
   - *2-е место*. `DS` - 2117 игр;
   - *3-е место*. `Wii` - 1275 игр;
   - *4-е место*. `PSP` - 1179 игр;
   - *5-е место*. `X360` - 1118 игр;
   - *6-е место*. `PS3` - 1087 игр;
   - *7-е место*. `GBA` - 810 игр.

Теперь выведем наш срез данных, краткую сводку по нему и абсолютное и относительное количество пропусков.

In [30]:
# Выведем первые 5 строк:
df_actual.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,user_score_category,critic_score_category
8212,THE ADVENTURES OF COOKIE & CREAM,PS2,2000,puzzle,0.09,0.07,0.0,0.02,NaN,NaN,NaN,NaN,NaN
12900,ANIMORPHS: SHATTERED REALITY,PS,2000,action,0.03,0.02,0.0,0.00,27.0,NaN,E,NaN,низкая оценка
7424,WILD WILD RACING,PS2,2000,racing,0.10,0.08,0.0,0.03,64.0,NaN,E,NaN,средняя оценка
7472,HBO BOXING,PS,2000,fighting,0.12,0.08,0.0,0.01,26.0,4.0,E,средняя оценка,низкая оценка
3620,NASCAR RUMBLE,PS,2000,racing,0.31,0.21,0.0,0.04,NaN,NaN,NaN,NaN,NaN


In [31]:
# Выведем сводную информацию:
df_actual.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12772 entries, 8212 to 15817
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   name                   12772 non-null  object  
 1   platform               12772 non-null  object  
 2   year_of_release        12772 non-null  int64   
 3   genre                  12772 non-null  object  
 4   na_sales               12772 non-null  float64 
 5   eu_sales               12772 non-null  float64 
 6   jp_sales               12772 non-null  float64 
 7   other_sales            12772 non-null  float64 
 8   critic_score           7162 non-null   float64 
 9   user_score             6476 non-null   float64 
 10  rating                 8715 non-null   object  
 11  user_score_category    6476 non-null   category
 12  critic_score_category  7162 non-null   category
dtypes: category(2), float64(6), int64(1), object(4)
memory usage: 1.2+ MB


In [32]:
# Считаем абсолютное значение пропусков (их сумму по столбцам):
final_missing_sum = df_actual.isna().sum()
# Считаем относительное значение (долю) пропусков и переводим в проценты:
final_missing_share = df_actual.isna().mean() * 100
# Округлим относительное значение:
final_missing_share = final_missing_share.round(2)
# Создадим датафрейм с пропусками для наглядности:
final_missing_df = pd.DataFrame({'Количество пропусков': final_missing_sum, 'Доля пропусков в процентах': final_missing_share})
# Выведем датафрейм:
final_missing_df

,Количество пропусков,Доля пропусков в процентах
name,0,0.00
platform,0,0.00
year_of_release,0,0.00
genre,0,0.00
na_sales,0,0.00
eu_sales,0,0.00
jp_sales,0,0.00
other_sales,0,0.00
critic_score,5610,43.92
user_score,6296,49.30


В итоге мы получили срез данных, который включает в себя период 2000-2013 гг. В нем представлены 13 столбцов и 12772 строки. Срез по-прежнему содержит пропуски, однако их удаление поспособствует искажению результатов дальнейшего исследования.

Поставленные перед нами задачи выполнены. Срез готов к последующему анализу.